# RJMCMC
Checking data stationarity by implementing RJMCMC analysis with change points.


In [ ]:
from torch.utils.data import ConcatDataset
import glob
import xarray as xr

basin_files = glob.glob("Caravan-nc/timeseries/netcdf/camels/*.nc")
basin_files = basin_files[:25]

In [6]:
import numpy as np
from scipy.stats import pearson3

def calc_return_periods(ds):
    '''
    Takes in nc ds and treats it using B17b return period fitting 
    Returns dict (water_year, return period of max streamflow event that year)
    '''

    annual_max = ds['streamflow'].groupby('water_year').max('date')
    annual_max = annual_max.where(annual_max > 0, drop=True)  # remove zero-flow years
    log_annual_max = np.log10(annual_max)

    # annual_max = ds['streamflow'].groupby('water_year').max('date')
    # log_annual_max = np.log10(annual_max) # For the B17b fitting procedure

    params = pearson3.fit(log_annual_max)
    skew, mu, sig = params # shape location scale of the fit

    # Calculate the return years by evaluating the fit PDF distribution
    p3cdf = pearson3.cdf(log_annual_max, skew, loc=mu, scale=sig)
    return_periods_list = 1/(1-p3cdf)

    # I want these to live in a dictionary
    return_periods = dict(zip(annual_max['water_year'].values, return_periods_list))

    return return_periods

def add_water_year(ds):
    '''
    Water year is defined as Oct 1 to  Sep 30.
    Oct, Nov, Dec are labelled as regular year + 1
    '''
    t = ds['date']
    water_year = t.dt.year + (t.dt.month >= 10)
    new_ds = ds.assign_coords(water_year=water_year)
    return new_ds

In [18]:
from torch.utils.data import ConcatDataset

multibasin_annual_maxima = [] # One entry for each basin

for basin_file in basin_files:
    ds = xr.open_dataset(basin_file)

    # Get rid of NaN values
    ds = ds.dropna(dim='date', 
               subset=['streamflow', 
                       'total_precipitation_sum',
                       'temperature_2m_mean',
                       'surface_net_solar_radiation_mean',
                       'surface_net_thermal_radiation_mean',
                       'dewpoint_temperature_2m_mean',
                       'u_component_of_wind_10m_mean',
                       'v_component_of_wind_10m_mean'], # From which features
                 how='any') # Any of the subsets

    # Water year
    ds = add_water_year(ds)

    # Put the max streamflow values into a list
    basin_annual_maxima = ds['streamflow'].groupby('water_year').max('date')
    basin_annual_maxima = basin_annual_maxima.where(basin_annual_maxima > 0, drop=True)  # remove zero-flow years
    basin_annual_maxima_dict = dict(zip(basin_annual_maxima['water_year'].values, basin_annual_maxima.values))
    multibasin_annual_maxima.append(basin_annual_maxima_dict)


In [ ]:
import numpy as np
from scipy.stats import gamma, poisson
from scipy.special import gammaln, logsumexp
from tqdm import tqdm
from copy import deepcopy

class RJMCMC:

    def __init__(self,
                        multibasin_annual_maxima, # a dictionary for each basin; each is a dict of annual maxima 
                        k_min, k_max, # the range over which k are uniform

                 ):
        self.multibasin_annual_maxima = multibasin_annual_maxima

        # Prior parameters

        # Uniform prior over k params
        self.k_min = k_min
        self.k_max = k_max

        # Normal priors over LP3 mean for each basin
        mu_mus = [] # means of the normal prior over lp3 mean
        std_mus = []

        # Normal priors over LP3 sig for each basin
        mu_sigs = []
        std_sigs = []

        for basin_annual_maxima in multibasin_annual_maxima:
            basin_vals = basin_annual_maxima.values()

            # mu_b,j has a prior for each basin; mu_mu given by empirical mean basin_annual_maxima and std_mu empircal spread
            mu_mu =  np.mean(basin_vals) # mean of the normal prior over lp3 mean
            std_mu = np.std(basin_vals) # std of ^
            mu_mus.append(mu_mu)
            std_mus.append(std_mu)

            # sig_b,j 
            

        
            
